# IMDB Sentiment Analysis

Text preprocessing ETL pipeline and baseline Keras classifier.

## 1. Dataset Ingestion & Cleaning

In [ ]:
import re
import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

## Dataset

Kaggle source: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [ ]:
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
zip_path = "imdb-dataset-of-50k-movie-reviews.zip"

if not os.path.exists("IMDB Dataset.csv"):
    urllib.request.urlretrieve(dataset_url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(".")
    os.remove(zip_path)

df = pd.read_csv("IMDB Dataset.csv")

print(df.shape)
display(df.head())
print(df.columns.tolist())
print(df.isnull().sum())

In [ ]:
df["sentiment"] = df["sentiment"].map({"positive": 1, "negative": 0})

print("Missing values:")
display(df[["review", "sentiment"]].isnull().sum())
print("\nClass distribution:")
display(df["sentiment"].value_counts().rename(index={0: "negative", 1: "positive"}))

In [ ]:
def clean_text(raw_text):
    text = re.sub(r"<.*?>", " ", str(raw_text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["cleaned_review"] = df["review"].apply(clean_text)
display(df[["review", "cleaned_review"]].head())

In [ ]:
df["raw_token_count"] = df["review"].str.split().str.len()
df["clean_token_count"] = df["cleaned_review"].str.split().str.len()

raw_vocab = set(" ".join(df["review"].astype(str)).lower().split())
clean_vocab = set(" ".join(df["cleaned_review"]).split())

summary = pd.DataFrame({
    "Metric": [
        "Average token count",
        "Median token count",
        "Total tokens",
        "Vocabulary size"
    ],
    "Raw": [
        df["raw_token_count"].mean(),
        df["raw_token_count"].median(),
        df["raw_token_count"].sum(),
        len(raw_vocab)
    ],
    "Cleaned": [
        df["clean_token_count"].mean(),
        df["clean_token_count"].median(),
        df["clean_token_count"].sum(),
        len(clean_vocab)
    ]
})

display(summary)

## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["cleaned_review"].astype(str).tolist(),
    df["sentiment"].astype("float32").tolist(),
    test_size=0.20,
    stratify=df["sentiment"],
    random_state=42
)

X_train = tf.constant(X_train, dtype=tf.string)
X_test = tf.constant(X_test, dtype=tf.string)
y_train = tf.constant(y_train, dtype=tf.float32)
y_test = tf.constant(y_test, dtype=tf.float32)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("X_train dtype:", X_train.dtype)
print("y_train dtype:", y_train.dtype)

## 3. TextVectorization

> **Fix note:** the original notebook cleaned training text with a python `clean_text()` function (stripping HTML tags/URLs), but embedded a *default* `TextVectorization` layer inside the deployed model for raw-string inference (Section 7). The default standardization only lowercases and strips punctuation — it does not remove HTML tags or URLs — so raw inputs like `"<br>...https://example.com"` were tokenized very differently from how the model's vocabulary was built, causing train/serve skew and unreliable predictions on raw text. The `TextVectorization` layer below now uses a TensorFlow-native `standardize` function that mirrors `clean_text()` exactly, so raw and training-time cleaning always match.

In [ ]:
def tf_clean_text(text):

    text = tf.strings.regex_replace(text, r"<.*?>", " ")
    text = tf.strings.regex_replace(text, r"https?://\S+|www\.\S+", " ")
    text = tf.strings.lower(text)
    text = tf.strings.regex_replace(text, r"[^a-zA-Z0-9\s']", " ")
    text = tf.strings.regex_replace(text, r"\s+", " ")
    text = tf.strings.strip(text)
    return text

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=10000,
    output_mode="int",
    output_sequence_length=200,
    standardize=tf_clean_text
)

vectorizer.adapt(X_train)

vocab = vectorizer.get_vocabulary()
vocab_summary = pd.DataFrame({
    "Index": range(10),
    "Token": vocab[:10]
})

display(vocab_summary)
print("Index 0:", repr(vocab[0]))
print("Index 1:", vocab[1])
print("Vocabulary size:", len(vocab))

## 4. End-to-End Model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,), dtype=tf.string, name="raw_text_input"),
    vectorizer,
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=32,
        mask_zero=True
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid", name="sentiment_output")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Training

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.20
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Test Evaluation

In [ ]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test)
).batch(32)

test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)

y_prob = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
y_true = y_test.numpy().astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"Test accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"]
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["negative", "positive"]
)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

## 7. Raw String Inference

In [ ]:
new_reviews = np.array([
    "<br>This movie was absolutely fantastic! https://example.com/review",
    "www.example.com Terrible movie, I wasted my time.",
    "<p>I really loved this film. The acting was amazing!</p>",
    "The story was boring and the movie was disappointing."
])

new_reviews_tf = tf.constant(new_reviews, dtype=tf.string)

predictions = model.predict(new_reviews_tf, verbose=0).ravel()

inference_results = pd.DataFrame({
    "Raw Review": new_reviews,
    "Prediction": ["positive" if p >= 0.5 else "negative" for p in predictions],
    "Probability": predictions
})

display(inference_results)